# Train an MLP for six emotions


In [17]:
import json
import sys
from pathlib import Path

import skops.io as sio
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'src' / 'load_data.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Start Jupyter from the 6Emotions project or a model folder.')
sys.path.insert(0, str(PROJECT_ROOT))
from src.load_data import load_model_data, TARGET_LABELS
from src.features import make_tfidf_features
from src.evaluate import save_confusion_matrices

RANDOM_STATE = 42
JOY_CAP = 12000
MLP_SETTINGS = {
    'hidden_layer_sizes': (512,),
    'activation': 'relu',
    'solver': 'adam',
    'alpha': 0.1,
    'batch_size': 256,
    'learning_rate_init': 3e-4,
    'max_iter': 20,
    'early_stopping': True,
    'validation_fraction': 0.1,
    'n_iter_no_change': 5,
}
RESULTS_DIR = PROJECT_ROOT / 'models' / 'mlp' / 'results'
ARTIFACT_DIR = PROJECT_ROOT / 'models' / 'mlp' / 'artifacts'

In [21]:
data = load_model_data(base_path=PROJECT_ROOT / 'data', random_state=RANDOM_STATE, joy_cap=JOY_CAP)
full_train_df = data['full_train']
full_test_df = data['full_test']

assert not full_train_df[['text', 'label']].isna().any().any()
assert not full_test_df[['text', 'label']].isna().any().any()
assert set(full_train_df['label']) == set(TARGET_LABELS)
assert set(full_test_df['label']) == set(TARGET_LABELS)
print(f'Training size: {len(full_train_df)}')
print(f'Test size:     {len(full_test_df)}')
print(full_train_df['label'].value_counts().reindex(TARGET_LABELS))

Training size: 39718
Test size:     5421
label
sadness      7966
joy         12000
love         3904
anger        7756
fear         2784
surprise     5308
Name: count, dtype: int64


## Model



In [19]:
model = Pipeline([
    ('features', make_tfidf_features()),
    ('mlp', MLPClassifier(**MLP_SETTINGS, random_state=RANDOM_STATE)),
])

## Fit and evaluate

In [20]:
model.fit(full_train_df['text'], full_train_df['label'])

y_true = full_test_df['label']
y_pred = model.predict(full_test_df['text'])
test_accuracy = float(accuracy_score(y_true, y_pred))
test_macro_f1 = float(f1_score(y_true, y_pred, labels=TARGET_LABELS, average='macro'))
report = classification_report(
    y_true, y_pred, labels=TARGET_LABELS, target_names=TARGET_LABELS,
    output_dict=True, zero_division=0,
)
print(f'MLP epochs: {model.named_steps["mlp"].n_iter_}')
print(f'MLP test accuracy: {test_accuracy:.4f}; macro F1: {test_macro_f1:.4f}')
print(classification_report(
    y_true, y_pred, labels=TARGET_LABELS, target_names=TARGET_LABELS,
    zero_division=0,
))

MLP epochs: 10
MLP test accuracy: 0.7558; macro F1: 0.7179
              precision    recall  f1-score   support

     sadness       0.73      0.78      0.75       882
         joy       0.82      0.84      0.83      2297
        love       0.70      0.62      0.66       417
       anger       0.68      0.69      0.69       909
        fear       0.83      0.63      0.72       311
    surprise       0.67      0.66      0.66       605

    accuracy                           0.76      5421
   macro avg       0.74      0.70      0.72      5421
weighted avg       0.76      0.76      0.75      5421



In [ ]:
cm = save_confusion_matrices(y_true, y_pred, TARGET_LABELS, RESULTS_DIR, 'MLP')

## Save model and results

In [22]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
model_path = ARTIFACT_DIR / '6emotions_model.skops'
sio.dump(model, model_path)
summary = {
    'model_type': 'TF-IDF + MLPClassifier',
    'random_state': RANDOM_STATE,
    'training_rows': len(full_train_df),
    'test_rows': len(full_test_df),
    'joy_cap': JOY_CAP,
    'mlp_settings': {**MLP_SETTINGS, 'hidden_layer_sizes': list(MLP_SETTINGS['hidden_layer_sizes'])},
    'epochs': int(model.named_steps['mlp'].n_iter_),
    'test_accuracy': test_accuracy,
    'test_macro_f1': test_macro_f1,
    'labels': TARGET_LABELS,
}
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'training_summary.json').write_text(json.dumps(summary, indent=2) + '\n')
(RESULTS_DIR / 'classification_report.json').write_text(json.dumps(report, indent=2) + '\n')
print(f'Model saved to {model_path}; results saved to {RESULTS_DIR}')

Model saved to /Users/hayden/HaydensOffice/Projects/6Emotions/models/mlp/artifacts/6emotions_model.skops; results saved to /Users/hayden/HaydensOffice/Projects/6Emotions/models/mlp/results
